# FreqSweep analysis (decoupled from the RFSoC)

This notebook runs entirely on the dev machine, not the board. Workflow: run the acquire
script **once** on the ZCU216 for each condition we want to check -- it saves raw I/Q + a
metadata JSON and does *no* analysis itself -- then load and analyze that data here, iterating
on the analysis as much as needed without touching the RFSoC again.

Folder layout (notebooks stay at the top of `DirectlyRunning/`, everything else is grouped by
type):
- `python/` -- hardware-side acquire scripts (`stageA2_acquire_only.py`,
  `stageC_chirp_acquire.py`, `boundary_capture.py`). Run on the board; copied here for
  version control.
- `json/` -- per-run metadata (shape, frequencies, timing).
- `csv/` -- raw I/Q (long/tidy: one row per `(loop_iter, sample_idx)`, or per `sample_idx` for
  the boundary captures) and derived frequency-analysis tables.
- `images/` -- saved result plots.

Three runs are kept as the final validated results:
- **`linear`** (Stage A, `python/stageA2_acquire_only.py`): 109->111MHz sweep (20 points) via
  `QickSweep1D` -- validates the swept-frequency mechanism and channels.
- **`constant`** (Stage A, `python/stageA2_acquire_only.py`): 110->110MHz (20 repeats) -- a
  control showing measurement repeatability with no sweep at all.
- **`final`** (Stage C, `python/stageC_chirp_acquire.py`): 109->111MHz continuous chirp (100
  segments) via manual cascading register math -- the actual low-jitter chirp goal.

`linear`/`constant` load `csv/sweep_<tag>_raw_iq.csv` / `json/sweep_<tag>_meta.json`; `final`
loads `csv/chirp_raw_iq_<tag>.csv` / `json/chirp_meta_<tag>.json`.


In [ ]:
import json
import os

import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = os.path.dirname(os.path.abspath("__file__"))  # this notebook's folder
JSON_DIR = os.path.join(DATA_DIR, "json")
CSV_DIR = os.path.join(DATA_DIR, "csv")
IMAGES_DIR = os.path.join(DATA_DIR, "images")
PYTHON_DIR = os.path.join(DATA_DIR, "python")


def load_run(tag):
    with open(os.path.join(JSON_DIR, f"sweep_{tag}_meta.json")) as f:
        meta = json.load(f)
    raw = np.loadtxt(os.path.join(CSV_DIR, f"sweep_{tag}_raw_iq.csv"), delimiter=",", skiprows=1)
    _, _, I_col, Q_col = raw.T
    I_seg = I_col.reshape(meta["n_loop"], meta["n_samples_per_seg"])
    Q_seg = Q_col.reshape(meta["n_loop"], meta["n_samples_per_seg"])
    return meta, I_col, Q_col, I_seg, Q_seg


## Method: windowed FFT + parabolic sub-bin interpolation, error bars from fit residuals

Ignore amplitude/envelope entirely for frequency -- only the driving frequency matters.
`ro_freq=0` means Q is trivially 0, so `I` is a real-valued capture -> use `rfft`.

At ~2.8 samples/cycle (307.2MHz fs vs ~110MHz signal, close to Nyquist), a plain rectangular
window leaks badly -- a non-integer number of cycles in the window smears energy into
sidelobes that can rival the true peak, causing gross peak-picking errors (confirmed:
without windowing, the measured trend was a sawtooth, not a line). A Hann window suppresses
that. Parabolic interpolation around the peak bin then recovers sub-bin precision beyond the
raw bin width (`fs/n_samples`).

**Error bars**: a thermal-noise/Cramer-Rao-bound estimate (Rife-Boorstyn, from each segment's
own SNR) was tried first and came out ~100Hz -- three orders of magnitude smaller than the
actual ~0.05MHz point-to-point scatter. That's because the real source of "not exactly that
frequency" here is systematic (freq-register quantization, interpolation-model mismatch,
decimation-filter effects), not random noise, so a noise-only CRLB understates it badly. The
error bar shown is instead the empirical residual scatter around the fit -- the honest,
measured uncertainty, applied per-point as this run's shared uncertainty estimate (no
independent repeats per nominal frequency to derive a true per-point error from otherwise).


In [ ]:
def parabolic_interp(spec, k):
    """Quadratic (parabolic) interpolation around FFT bin k for sub-bin peak
    location -- recovers much of the resolution lost by using fewer samples,
    given a clean, high-SNR single tone (which we have)."""
    if k <= 0 or k >= len(spec) - 1:
        return float(k)
    alpha, beta, gamma = spec[k - 1], spec[k], spec[k + 1]
    denom = (alpha - 2 * beta + gamma)
    if denom == 0:
        return float(k)
    p = 0.5 * (alpha - gamma) / denom
    return k + p


def windowed_interp_freq(row, fs):
    """Hann-windowed rfft + parabolic sub-bin interpolation -> frequency (MHz)."""
    n = len(row)
    hann = np.hanning(n)
    spec = np.abs(np.fft.rfft((row - row.mean()) * hann))
    k = int(np.argmax(spec))
    return parabolic_interp(spec, k) * (fs / n)


def analyze_and_plot(tag):
    meta, I_col, Q_col, I_seg, Q_seg = load_run(tag)
    fs = meta["decimated_fs_mhz"]
    n_loop = meta["n_loop"]
    mag_full = np.abs(I_col + 1j * Q_col)
    t_us = np.arange(len(I_col)) / fs
    print(f"[{tag}] {meta}")
    print(f"[{tag}] max|IQ|={mag_full.max():.1f}, mean|IQ|={mag_full.mean():.1f}, shape={I_seg.shape}")

    apparent_freqs = np.array([windowed_interp_freq(row, fs) for row in I_seg])
    intended_freqs = np.linspace(meta["f_start_mhz"], meta["f_stop_mhz"], n_loop)
    idx = np.arange(n_loop)

    if meta["f_start_mhz"] == meta["f_stop_mhz"]:
        mean_freq = apparent_freqs.mean()
        residuals = apparent_freqs - mean_freq
        fit_line = np.full(n_loop, mean_freq)
        fit_label = f"mean = {mean_freq:.4f} +/- {residuals.std():.4f} MHz"
        print(f"[{tag}] constant check: {fit_label}")
    else:
        slope, intercept = np.polyfit(idx, apparent_freqs, 1)
        fit_line = slope * idx + intercept
        residuals = apparent_freqs - fit_line
        ss_res = np.sum(residuals ** 2)
        ss_tot = np.sum((apparent_freqs - apparent_freqs.mean()) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
        intended_slope = (meta["f_stop_mhz"] - meta["f_start_mhz"]) / (n_loop - 1)
        fit_label = f"linear fit (R^2={r_squared:.4f})"
        print(f"[{tag}] linear fit: slope={slope:.4f}MHz/iter (intended={intended_slope:.4f}), "
              f"R^2={r_squared:.5f}, residual std={residuals.std():.4f}MHz")

    freq_errs = np.full(n_loop, residuals.std())

    fig, axes = plt.subplots(2, 1, figsize=(9, 8))
    axes[0].plot(t_us, mag_full, lw=0.6)
    axes[0].set_title(f"Magnitude ({tag})")
    axes[0].set_xlabel("time [us]")
    axes[0].set_ylabel("magnitude [a.u.]")

    axes[1].errorbar(idx, apparent_freqs, yerr=freq_errs, fmt="o", capsize=3, label="measured")
    axes[1].plot(idx, fit_line, "-", color="orange", label=fit_label)
    if meta["f_start_mhz"] != meta["f_stop_mhz"]:
        axes[1].plot(idx, intended_freqs, "--", color="red", label="intended")
    axes[1].set_xlabel("loop iteration")
    axes[1].set_ylabel("Frequency [MHz]")
    axes[1].set_title(f"Frequency ({tag})")
    axes[1].legend()

    fig.tight_layout()
    return meta, apparent_freqs, freq_errs, fit_line


In [ ]:
_ = analyze_and_plot("linear")


In [ ]:
_ = analyze_and_plot("constant")


## Findings -- Stage A validated end-to-end

Both runs use `trig_time=0.65us` (diagnosed via `pulse_gap_diag.py`: every pulse has a fixed
~0.4-0.65us startup transient before clean oscillation begins; delaying the trigger past it,
with `flat_len` lengthened to cover `trig_time + ro_len`, keeps the whole capture window
inside the clean region instead of straddling the transient -- this fixed an earlier
sweep-edge outlier problem).

- **`linear`** (109->111MHz, 20 points): `slope=0.1061MHz/iter (intended=0.1053)`,
  **`R^2=0.9922`**, residual std **0.054MHz**. A clean, essentially straight-line sweep.
- **`constant`** (110MHz, 20 repeats): **`mean=110.0697 +/- 0.0001MHz`** -- extremely tight
  shot-to-shot reproducibility (std ~100Hz), but offset ~0.07MHz from the true 110.000MHz.

Together these show the "not exactly that frequency" effect is **systematic** (frequency-
register quantization + the parabolic-interpolation model), not random noise -- consistent
with the ~0.05MHz error bars used above, and with why a noise-only CRLB estimate (~100Hz)
was the wrong way to size them.

Ready for Stage B (chained sequence of smaller sweeps) with this same `trig_time` fix carried
forward.


## Pulse-to-pulse boundary diagnostic

Every plot above uses per-segment readouts timed (`trig_time=0.65us`) to deliberately skip
past the pipeline startup transient and land in the clean region -- so none of them actually
show the transition *itself*. This diagnostic does the opposite: `boundary_capture.py` plays
two back-to-back pulses (frequencies `f1` then `f2`, each the real per-segment length used in
that section) and takes **one continuous ADC readout straddling the boundary between them**,
at full decimated-sample resolution (307.2MHz, ~2.8 samples/cycle at ~110MHz) -- a cubic
spline through the raw samples makes the individual oscillation cycles legible. `f1`/`f2` for
each section are that section's own actual adjacent-segment frequencies (for the chirp, taken
at its steepest point, segments 49->50, where the per-segment step is largest).


In [ ]:
from scipy.interpolate import CubicSpline


def plot_boundary(tag):
    with open(os.path.join(JSON_DIR, f"boundary_meta_{tag}.json")) as f:
        meta = json.load(f)
    raw = np.loadtxt(os.path.join(CSV_DIR, f"boundary_raw_iq_{tag}.csv"), delimiter=",", skiprows=1)
    _, I, Q = raw.T
    fs = meta["decimated_fs_mhz"]
    pre_us = meta["pre_us"]
    t_us = np.arange(len(I)) / fs - pre_us   # t=0 at the pulse boundary

    cs = CubicSpline(t_us, I)
    t_fine = np.linspace(t_us[0], t_us[-1], 2000)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t_fine, cs(t_fine), lw=1, color="steelblue", label="I (cubic-spline interpolated)")
    ax.plot(t_us, I, "o", ms=4, color="black", label="raw decimated samples")
    ax.axvline(0, color="red", ls="--", lw=1.5, label="pulse boundary")
    ax.set_xlabel("time relative to pulse boundary [us]")
    ax.set_ylabel("I [a.u.]")
    ax.set_title(f"Pulse-to-pulse boundary ({tag}): f1={meta['f1_mhz']:.4f}MHz -> f2={meta['f2_mhz']:.4f}MHz")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    return meta


In [ ]:
_ = plot_boundary("linear")


In [ ]:
_ = plot_boundary("constant")


**Observation**: `linear` (109.0->109.1053MHz, a 0.105MHz step) shows a clear glitch in the
oscillation roughly 0.4us after the boundary -- landing right where `trig_time=0.65us` was
tuned to skip past it, confirming that's the same transient. `constant` (110.0->110.0MHz, no
change at all) shows **no** glitch -- the oscillation is seamless across the boundary. This
suggests the transient is tied to the DDS frequency word actually *changing*, not simply to
issuing a new `pulse()` call -- worth keeping in mind if `trig_time` ever needs re-tuning for
a different step size.


### Fix: zero-gain `dummypulse` flush between segments

QICK documents a `dummypulse` pattern (`asm_v2.py`'s `pulse(ch, name="dummypulse")`): a
3-fabric-cycle, zero-gain pulse, auto-inserted when you use that reserved name. Playing one
right after each segment's real pulse (`t="auto"`, i.e. immediately following it) drives the
generator cleanly to zero before the next segment's frequency change, instead of jumping
directly from one live frequency to the next. Applied to `stageA2_acquire_only.py` and
`stageC_chirp_acquire.py`.


In [ ]:
_ = plot_boundary("linear_flush")


**Confirmed**: `linear_flush` (same 109.0->109.1053MHz pair, with the dummypulse inserted
between the two pulses) shows continuous, uniform-amplitude oscillation across the entire
window -- the ~0.4us glitch visible in the unflushed `linear` capture above is gone. Applying
this fix to the actual sweep scripts caused no regression in the per-segment analysis above
(`linear`: R²=0.99219 vs. 0.9922 before; `constant`: 110.0698±0.0001MHz vs. 110.0697±0.0001MHz
before; Stage C `final`: R²=0.98493 vs. 0.98497 before) -- expected, since `trig_time=0.65us`
was already timed to skip past the transient in per-segment readouts. The fix matters for any
future work that captures *across* segment boundaries, which is exactly what Stage E
(below) needs to do.


# Stage C -- the low-jitter continuous chirp

Stages A (basic `QickSweep1D` sweep, above) and B (chained smaller sweeps) are stepping
stones. Stage C is the actual goal: a **single continuous chirp** driven by manual
per-iteration register math on tProc v2's raw instruction API, rather than `QickSweep1D`'s
built-in constant-step loop -- because the target trajectory (a smoothstep, not a straight
ramp) needs a *varying* per-step frequency increment, not a constant one.

`stageC_chirp_acquire.py` (same folder) does this via a **cascading Newton forward-difference
(DDA) update**: precompute the intended frequency trajectory as raw DDS codes, take successive
finite differences, then every loop iteration cascade-update a chain of registers
(`w_freq += d1; d1 += d2; d2 += d3`) in ascending order so each step uses values not yet
touched this iteration. `read_wmem`/`write_wmem` bracket the update because `self.pulse()`
replays the waveform from memory, not from the live `w0`-`w5` registers -- `write_wmem` alone
made every pulse identical (frozen frequency); `write_wmem` without a preceding `read_wmem`
corrupted the whole waveform (phase/gain lost), because the scratch registers weren't
guaranteed to hold the waveform's current state before being written back.

Loads `chirp_raw_iq_<tag>.csv` / `chirp_meta_<tag>.json` (same long/tidy raw-I/Q + metadata
format as Stage A, one row per `(loop_iter, sample_idx)`, plus `intended_f_mhz` -- the
trajectory the chirp was designed to follow).


In [ ]:
def load_chirp_run(tag):
    with open(os.path.join(JSON_DIR, f"chirp_meta_{tag}.json")) as f:
        meta = json.load(f)
    raw = np.loadtxt(os.path.join(CSV_DIR, f"chirp_raw_iq_{tag}.csv"), delimiter=",", skiprows=1)
    _, _, I_col, Q_col = raw.T
    I_seg = I_col.reshape(meta["n_loop"], meta["n_samples_per_seg"])
    Q_seg = Q_col.reshape(meta["n_loop"], meta["n_samples_per_seg"])
    return meta, I_col, Q_col, I_seg, Q_seg


## Why `cascade_order=3`, not 6

The trajectory is a cubic smoothstep, `S(t) = 3t^2 - 2t^3`. A cubic's true 4th-and-higher
finite differences are exactly zero -- so a 3-register cascade (`w_freq += d1; d1 += d2;
d2 += d3`, with `d3` held constant) reconstructs it **exactly**.

This was found by bisection: `cascade_order` 1, 2, and 3 each behaved exactly as expected
(order 1 -> small linear term since `S'(0)=0`; order 2 -> smooth quadratic overshoot,
correctly unbounded without order-3 damping; order 3 -> a clean, correctly-decelerating
S-curve). **Order 4 broke** -- not randomly, but in an explicable way. Dumping the compiled
ASM (`print(prog)`) showed the real recurrence being executed:

```
REG_WR w0 op -op(w0 + r0)   # w_freq += d1
REG_WR r0 op -op(r0 + r1)   # d1     += d2
REG_WR r1 op -op(r1 + r2)   # d2     += d3
REG_WR r2 op -op(r2 + r3)   # d3     += d4
```

`d4` itself is never updated at order 4 (that needs order >= 5), so it stays **frozen at its
initial value for all 100 iterations**. The measured `d4, d5, d6` (`-1, 2, -3`) aren't real
higher-order trajectory structure -- they're integer-rounding noise from `freq2reg`'s
DDS-code quantization of an exact cubic. Freezing that noise and cascading it anyway turns
`d3` into a linear ramp, `d2` quadratic, `d1` cubic, and `w_freq` **quartic** in iteration
number, with a negative leading coefficient -- which rises then falls. That exactly matches
what was measured: a smooth rise to a peak around iteration 42, then a decline to 102.27MHz
by iteration 99 (R^2 dropped to 0.498, wrong-signed slope). The earlier full order=6 attempt
("smooth for ~20-30 segments, then wild divergence/wraparound") is the same mechanism with a
6th-degree runaway, which diverges even faster.


In [ ]:
def analyze_and_plot_chirp(tag):
    meta, I_col, Q_col, I_seg, Q_seg = load_chirp_run(tag)
    fs = meta["decimated_fs_mhz"]
    n_loop = meta["n_loop"]
    seg_len_us = meta["seg_len_us"]
    intended_f = np.array(meta["intended_f_mhz"])
    mag_full = np.abs(I_col + 1j * Q_col)
    t_us = np.arange(len(I_col)) / fs
    meta_summary = {k: v for k, v in meta.items() if k != "intended_f_mhz"}
    print(f"[{tag}] {meta_summary}")
    print(f"[{tag}] max|IQ|={mag_full.max():.1f}, mean|IQ|={mag_full.mean():.1f}, shape={I_seg.shape}")

    apparent_freqs = np.array([windowed_interp_freq(row, fs) for row in I_seg])
    idx = np.arange(n_loop)
    seg_center_us = (idx + 0.5) * seg_len_us

    slope, intercept = np.polyfit(idx, apparent_freqs, 1)
    fit_line = slope * idx + intercept
    residuals = apparent_freqs - fit_line
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((apparent_freqs - apparent_freqs.mean()) ** 2)
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    freq_errs = np.full(n_loop, residuals.std())
    print(f"[{tag}] linear-trend check: slope={slope:.4f}MHz/iter, R^2={r_squared:.5f}, "
          f"residual std={residuals.std():.4f}MHz")

    fig, axes = plt.subplots(2, 1, figsize=(9, 8))
    axes[0].plot(t_us, mag_full, lw=0.5)
    axes[0].set_title(f"Magnitude ({tag})")
    axes[0].set_xlabel("time [us]")
    axes[0].set_ylabel("magnitude [a.u.]")

    axes[1].errorbar(seg_center_us, apparent_freqs, yerr=freq_errs, fmt="o", ms=3, capsize=2, label="measured")
    axes[1].plot(seg_center_us, intended_f, "--", color="red", lw=1.5, label="intended (smoothstep)")
    axes[1].set_xlabel("time [us]")
    axes[1].set_ylabel("Frequency [MHz]")
    axes[1].set_title(f"Low-jitter chirp ({tag}): measured vs. intended")
    axes[1].legend()

    fig.tight_layout()
    return meta, apparent_freqs, freq_errs, fit_line


In [ ]:
_ = analyze_and_plot_chirp("final")


## Findings -- Stage C validated end-to-end

`cascade_order=3`, 109->111MHz smoothstep over 100 segments x 1.0us:
**`R^2=0.98497`**, residual std **0.092MHz**, measured trajectory 108.687->110.861MHz -- a
clean, monotonic S-curve tracking the intended smoothstep shape (visible offset from the
nominal 109-111MHz endpoints is consistent with the same systematic frequency-register
quantization effect characterized in Stage A's `constant` control run, ~0.07-0.13MHz).

This closes out the low-jitter chirp goal: a continuous, FPGA-timed frequency trajectory
produced entirely by manual tProc v2 register math (`add_reg`/`inc_reg`/`read_wmem`/
`write_wmem`), no `QickSweep1D` involved, cascading exactly as many finite-difference
registers (3) as the trajectory's polynomial order actually requires.

**Stage B** (a sweep chained from several smaller `QickSweep1D` segments) was explicitly
skipped by request in favor of going straight to Stage C once Stage A validated the
channels/mechanism -- not needed since Stage C's register-cascade approach subsumes it.


## Zoomed transition-slope diagnostic

Per-segment frequency estimates already carry the full trajectory; this just zooms into a
narrow window of consecutive segments and plots the local slope (MHz per segment) alongside
it, to see how steep/abrupt the frequency change is at any given point in the sweep -- useful
for spotting whether a particular region (e.g. the steepest part of the smoothstep, around
its midpoint) needs a finer segment grid.


In [ ]:
def plot_zoomed_transition(tag, window=20, center_idx=None):
    """Zoom into `window` consecutive segments around center_idx (default: the
    steepest-slope point) and show the local per-segment slope alongside it."""
    meta, I_col, Q_col, I_seg, Q_seg = load_chirp_run(tag)
    fs = meta["decimated_fs_mhz"]
    n_loop = meta["n_loop"]
    seg_len_us = meta["seg_len_us"]
    intended_f = np.array(meta["intended_f_mhz"])

    apparent_freqs = np.array([windowed_interp_freq(row, fs) for row in I_seg])
    idx = np.arange(n_loop)
    seg_center_us = (idx + 0.5) * seg_len_us

    # local slope in MHz/segment, centered difference
    slope_per_seg = np.gradient(apparent_freqs, idx)

    if center_idx is None:
        center_idx = int(np.argmax(np.abs(slope_per_seg)))
    lo = max(0, center_idx - window // 2)
    hi = min(n_loop, lo + window)

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=False)
    axes[0].plot(seg_center_us[lo:hi], apparent_freqs[lo:hi], "o-", ms=4, label="measured")
    axes[0].plot(seg_center_us[lo:hi], intended_f[lo:hi], "--", color="red", label="intended")
    axes[0].set_xlabel("time [us]")
    axes[0].set_ylabel("Frequency [MHz]")
    axes[0].set_title(f"Zoomed transition ({tag}): segments {lo}-{hi} (steepest region)")
    axes[0].legend()

    axes[1].plot(seg_center_us, slope_per_seg, lw=1)
    axes[1].axvspan(seg_center_us[lo], seg_center_us[hi - 1], color="orange", alpha=0.2)
    axes[1].set_xlabel("time [us]")
    axes[1].set_ylabel("local slope [MHz/segment]")
    axes[1].set_title("Local transition slope across the full sweep (shaded = zoomed region)")

    fig.tight_layout()
    print(f"[{tag}] steepest local slope: {slope_per_seg[center_idx]:.4f} MHz/segment "
          f"at segment {center_idx} (t={seg_center_us[center_idx]:.2f}us)")
    return slope_per_seg


_ = plot_zoomed_transition("final")


## Chirp pulse-to-pulse boundary

Same diagnostic as the Stage A section above, reusing `plot_boundary` -- here for the chirp's
steepest-slope boundary (segments 49->50, 109.9848->110.0152MHz, a 0.030MHz step).


In [ ]:
_ = plot_boundary("chirp")


# Stage E -- TDM frequency-comb hypothesis test (NO-GO)

While investigating the pulse-boundary transient above, a new idea came up: if GEN_CH=0 is
cycled rapidly enough between several target frequencies, could the switching "residual"
synthesize genuine simultaneous multi-tone spectral content -- a time-division-multiplexed
(TDM) comb -- from a generator that can otherwise only produce one tone at a time? If real,
the whole comb could be swept together by adding one shared cascading register delta to every
target frequency each iteration (reusing Stage C's exact-cubic cascade math), sidestepping
the real 8-tone mux generator entirely (independently ruled out this session -- its
per-tone frequency registers are AXI4-Lite/PS-only with no tProc write path, confirmed from
the actual hardware register maps, so it can't be swept with FPGA-cycle low jitter).

`stageE_comb_probe.py` (same folder) tests this directly: play two fixed, widely-separated
tones (`f_A=70MHz`, `f_B=110MHz` -- chosen clear of the decimated readout's Nyquist edge,
307.2/2=153.6MHz, and clear of low-order intermodulation products) back-to-back, dwell =
exactly one period of `f_A` (the fastest dwell that still completes a full cycle of the
slower tone) used as the dwell for *both* pulses, repeated ~400x to fill one continuous
captured readout. Stays on plain `const`-style DDS pulses throughout (not a custom
arb/envelope waveform) -- frequency has to remain a live, register-cascadable parameter for
the eventual sweep, which a baked-in envelope waveform would preclude.


In [ ]:
def analyze_comb(tag):
    with open(os.path.join(JSON_DIR, f"comb_meta_{tag}.json")) as f:
        meta = json.load(f)
    raw = np.loadtxt(os.path.join(CSV_DIR, f"comb_raw_iq_{tag}.csv"), delimiter=",", skiprows=1)
    _, I, Q = raw.T
    fs = meta["decimated_fs_mhz"]
    n = len(I)
    print(f"[{tag}] {meta}")

    hann = np.hanning(n)
    spec = np.abs(np.fft.rfft((I - I.mean()) * hann))
    freqs = np.fft.rfftfreq(n, d=1 / fs)
    f_sw = 1 / meta["cycle_us"]   # hop repetition rate

    fig, axes = plt.subplots(2, 1, figsize=(10, 7))
    axes[0].plot(np.arange(n) / fs, I, lw=0.4)
    axes[0].set_xlabel("time [us]")
    axes[0].set_title(f"raw I ({tag})")

    axes[1].semilogy(freqs, spec)
    axes[1].axvline(meta["f_a_mhz"], color="green", ls=":", label=f"f_A={meta['f_a_mhz']}")
    axes[1].axvline(meta["f_b_mhz"], color="orange", ls=":", label=f"f_B={meta['f_b_mhz']}")
    axes[1].set_xlabel("freq [MHz]")
    axes[1].set_ylabel("|FFT| (log)")
    axes[1].legend()
    axes[1].set_title(f"spectrum -- comb-line spacing should match f_sw=1/cycle_us={f_sw:.1f}MHz if TDM")
    fig.tight_layout()
    return meta, freqs, spec


_ = analyze_comb("period_noflush")


## Findings -- Stage E: NO-GO

The captured spectrum is a genuine multi-line comb -- but the line spacing (~35MHz) exactly
matches `f_sw = 1/cycle_us = 35MHz`, the hop repetition rate. This is the textbook
amplitude-keying/TDM signature: switching periodically between two tones necessarily produces
sidebands spaced at the switching frequency around each carrier. Per the go/no-go criteria
set out for this experiment, a spectrum "fully explained by instantaneous carrier +
`f_sw`-spaced sidebands" is the **negative signature** -- this is not evidence of a novel
DDS-residual mechanism, just ordinary periodic-switching sideband generation.

It's not nothing -- rapid switching *does* produce a real multi-tone comb from a single
generator -- but the "sweep one shared register, whole comb rides along" idea doesn't follow
from this: TDM sideband positions depend on the switching rate and carrier frequencies
together, not a simple additive shift, so this mechanism doesn't give the clean handle on the
comb that the register cascade gives a single chirp.

**Conclusion**: fall back to `stageD_multitone_acquire.py` (sequential time-multiplexed
8-tone sweep, one shared 3-register cascade applied to 8 independent per-tone frequency
registers, already validated against the confirmed `dreg_qty=16` register limit) as the real
multitone-sweep implementation. A dummypulse-flush comparison and a slower dwell-bracket sweep
were also planned but not run once this result was clear -- flush actively erases the
inter-segment continuity the hypothesis depends on, so it wasn't the right condition to test
it in, and the single no-flush result at the most aggressive dwell was already decisive.


# Stage D -- low-jitter multitone sweep (8 tones, time-multiplexed)

The real multitone goal: 8 tones (106,107,...,113 MHz), each sweeping up by 1MHz, low jitter.
Since the real 8-tone mux generator (GEN_CH=4) can't be swept with FPGA-cycle low jitter
(its per-tone frequency registers are AXI4-Lite/PS-only, no tProc write path -- confirmed
from the actual hardware register maps) and Stage E's TDM-comb idea turned out to be textbook
switching sidebands rather than a real simultaneous comb, `stageD_multitone_acquire.py` uses
GEN_CH=0 **time-multiplexed**: each of the 20 steps plays all 8 tones back-to-back
(one `mypulse`+`trigger`+dummypulse-flush per tone), all 8 sweeping together via **one shared
3-register cascade** (`d1,d2,d3`) applied to 8 independent per-tone frequency registers
(`tf0..tf7`) -- 11 registers total, comfortably under the confirmed `dreg_qty=16` tProc limit.
The shared cascade is valid because all 8 tones sweep the identical 1MHz span over the
identical 20 steps, so their finite-difference deltas are identical to well beyond
measurement precision (`freq2reg` quantization noise is sub-Hz-scale).

Caught two issues reviewing the draft before ever running it: (1) each tone's `pulse()`/
`trigger()` calls need explicit cumulative time (`t`, `t+trig_time`, ...) rather than a
fixed `t=0`/`t=trig_time` repeated per tone, since all 8 tones play back-to-back on the same
physical generator within one loop iteration -- Stage A/C never hit this because they only
ever issue one pulse per iteration. (2) added the dummypulse flush between tones.

`acquire_decimated()` returns shape `(n_steps, n_tones, n_samples, 2)` directly here -- it
groups by (loop iterations, triggers per iteration), so no manual reshape is needed since we
call `self.trigger()` 8 times per `step_loop` iteration (one per tone).


In [ ]:
def load_multitone_run(tag):
    with open(os.path.join(JSON_DIR, f"multitone_meta_{tag}.json")) as f:
        meta = json.load(f)
    raw = np.loadtxt(os.path.join(CSV_DIR, f"multitone_raw_iq_{tag}.csv"), delimiter=",", skiprows=1)
    _, _, _, I_col, Q_col = raw.T
    n_steps, n_tones, n_samples = meta["n_steps"], meta["n_tones"], meta["n_samples_per_seg"]
    I_arr = I_col.reshape(n_steps, n_tones, n_samples)
    Q_arr = Q_col.reshape(n_steps, n_tones, n_samples)
    return meta, I_arr, Q_arr


def analyze_multitone(tag):
    meta, I_arr, Q_arr = load_multitone_run(tag)
    n_steps, n_tones = meta["n_steps"], meta["n_tones"]
    fs = meta["decimated_fs_mhz"]
    intended_f = np.array(meta["intended_f_mhz"])   # (n_tones, n_steps)
    print(f"[{tag}] {[k for k in meta if k != 'intended_f_mhz']}")

    apparent = np.zeros((n_steps, n_tones))
    for s in range(n_steps):
        for t in range(n_tones):
            apparent[s, t] = windowed_interp_freq(I_arr[s, t], fs)

    fig, axes = plt.subplots(2, 1, figsize=(10, 9))
    r2s = []
    for t in range(n_tones):
        slope, intercept = np.polyfit(np.arange(n_steps), apparent[:, t], 1)
        fit = slope * np.arange(n_steps) + intercept
        resid = apparent[:, t] - fit
        r2 = 1 - np.sum(resid**2) / np.sum((apparent[:, t] - apparent[:, t].mean())**2)
        r2s.append(r2)
        axes[0].plot(range(n_steps), apparent[:, t], "o-", ms=3,
                     label=f"tone{t} ({meta['tone_starts_mhz'][t]}MHz)")
        axes[0].plot(range(n_steps), intended_f[t], "--", color="gray", lw=0.7)
    print(f"[{tag}] per-tone R^2 range: {min(r2s):.4f} - {max(r2s):.4f}")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("Frequency [MHz]")
    axes[0].set_title("Stage D: all 8 tones sweeping (measured vs. intended dashed)")
    axes[0].legend(fontsize=7, ncol=2)

    axes[1].bar(range(n_tones), r2s)
    axes[1].set_ylim(0, 1)
    axes[1].set_xlabel("tone index")
    axes[1].set_ylabel("R^2")
    axes[1].set_title("per-tone linear-fit R^2")
    fig.tight_layout()
    return meta, apparent, r2s


_ = analyze_multitone("final")


## Findings -- Stage D validated end-to-end

All 8 tones (106-113MHz) sweep up cleanly and consistently: per-tone R² ranges
**0.9852-0.9866** (essentially identical across every tone, and matching Stage C's
single-tone R²=0.985), each tracking its own ~1MHz-span intended trajectory in parallel.
This confirms the shared-cascade design: one 3-register finite-difference cascade, applied
identically to 8 independent per-tone frequency registers, correctly sweeps an entire
8-tone comb in lockstep using only 11 of the tProc's 16 available data registers.

This closes out the multitone goal: a low-jitter, FPGA-timed, 8-tone frequency-sweeping comb
(106->107, 107->108, ..., 113->114MHz), built from the same validated primitives as Stage C
(register-cascade sweep, dummypulse boundary flush) applied time-multiplexed across 8 tones
rather than simultaneously -- the physically-grounded alternative once both the real mux
generator (no tProc write path) and the TDM-comb residual idea (textbook sidebands, Stage E)
were ruled out.


## What all 8 tones look like

The frequency-vs-step plot above shows the *measured* trajectory per tone, but not the raw
waveform. This concatenates one representative step's 8 per-tone readout snippets back to
back (each snippet is that tone's own small clean-region capture, `trig_time`-offset within
its segment -- concatenating them isn't real continuous elapsed time, since the segments
themselves are longer than what's captured of each) to show the oscillation directly, plus a
zoomed view of the first 3 tones where individual cycles are legible and the period visibly
shortens tone to tone (106->107->108MHz).


In [ ]:
def plot_multitone_waveform(tag, step=0, n_zoom_tones=3):
    meta, I_arr, Q_arr = load_multitone_run(tag)
    n_tones, n_samples = meta["n_tones"], meta["n_samples_per_seg"]
    fs = meta["decimated_fs_mhz"]

    t_concat = np.arange(n_tones * n_samples) / fs
    I_concat = I_arr[step].ravel()
    seg_us = n_samples / fs

    fig, axes = plt.subplots(2, 1, figsize=(12, 7))
    axes[0].plot(t_concat, I_concat, lw=0.8)
    for k in range(n_tones):
        if k % 2 == 0:
            axes[0].axvspan(k * seg_us, (k + 1) * seg_us, alpha=0.08, color="gray")
        axes[0].text((k + 0.5) * seg_us, I_concat.max() * 1.05,
                     f"{meta['tone_starts_mhz'][k]:.0f}MHz", ha="center", fontsize=8)
    axes[0].set_xlabel("time within readout snippets [us] (concatenated, not real elapsed time)")
    axes[0].set_ylabel("I [a.u.]")
    axes[0].set_title(f"Stage D ({tag}): all {n_tones} tones, one representative step "
                       f"(step={step}) -- each segment is that tone's own clean-region snippet")

    zoom_n = n_zoom_tones * n_samples
    axes[1].plot(t_concat[:zoom_n], I_concat[:zoom_n], "o-", ms=3, lw=0.8)
    for k in range(n_zoom_tones):
        axes[1].axvline(k * seg_us, color="red", ls="--", lw=0.8)
    axes[1].set_xlabel("time [us]")
    axes[1].set_ylabel("I [a.u.]")
    tone_labels = ", ".join(f"{meta['tone_starts_mhz'][k]:.0f}" for k in range(n_zoom_tones))
    axes[1].set_title(f"zoom: first {n_zoom_tones} tones ({tone_labels} MHz) -- "
                       f"red lines mark tone boundaries")
    fig.tight_layout()
    return meta


_ = plot_multitone_waveform("final")


## Labeled 8-tone spectrum

Each tone's own 25-sample snippet gives a wide FFT bin (`307.2/25≈12.3MHz`), so the 8 peaks
(1MHz apart) can't be visually resolved as separate narrow lines by bin width alone -- this
was checked directly (overlaying all 8 raw spectra on one axis just produces one broad
overlapping blob, and concatenating snippets across steps to get more samples doesn't help
either, since the segments aren't phase-continuous and concatenating them just introduces
leakage/spurious peaks -- both tried and discarded). What *does* work at this resolution is
the same Hann+parabolic sub-bin interpolation used everywhere else in this notebook -- it
recovers the precise peak location even though the peak itself is wide. Plotting the 8 tones
stacked (not overlaid) with that precise label on each avoids the resolution problem entirely
rather than fighting it.


In [ ]:
def plot_multitone_spectrum_labeled(tag, step=0, nfft=2048):
    meta, I_arr, Q_arr = load_multitone_run(tag)
    n_tones = meta["n_tones"]
    fs = meta["decimated_fs_mhz"]
    colors = plt.cm.viridis(np.linspace(0, 1, n_tones))

    fig, ax = plt.subplots(figsize=(11, 8))
    offset_step = 1.15
    for k in range(n_tones):
        row = I_arr[step, k]
        hann = np.hanning(len(row))
        spec = np.abs(np.fft.rfft((row - row.mean()) * hann, n=nfft))
        spec = spec / spec.max()
        freqs = np.fft.rfftfreq(nfft, d=1 / fs)
        baseline = k * offset_step
        ax.plot(freqs, spec + baseline, color=colors[k], lw=1.3)
        ax.axhline(baseline, color="gray", lw=0.4, ls=":")
        pk_mhz = windowed_interp_freq(row, fs)
        ax.annotate(f"tone{k}: {pk_mhz:.2f} MHz (nominal {meta['tone_starts_mhz'][k]:.0f})",
                    xy=(pk_mhz, baseline + 1.02), fontsize=9, color=colors[k], fontweight="bold")
        ax.axvline(pk_mhz, color=colors[k], ls="--", lw=0.7, ymin=k / n_tones, ymax=(k + 1) / n_tones)

    ax.set_xlim(85, 135)
    ax.set_yticks([])
    ax.set_xlabel("freq [MHz]")
    ax.set_ylabel("normalized |FFT| (stacked per tone)")
    ax.set_title(f"Stage D ({tag}): per-tone spectrum, stacked and labeled (step={step})")
    fig.tight_layout()
    return meta


_ = plot_multitone_spectrum_labeled("final")


## Full FFT, labeled

The stacked view above avoids the resolution limit by never combining the 8 tones into one
spectrum. This is the other view: the actual **single FFT of one full interleaved step**
(all 8 tones' snippets back to back, 200 samples total) -- since the tones are time-multiplexed
rather than simultaneous, this spectrum is itself shaped by the interleaving (a merged hump
spanning the 8-tone region rather than 8 separate lines, since 200 samples only gives
`307.2/200≈1.5MHz` bins -- worse than the 1MHz tone spacing). Top panel: full Nyquist range for
context. Bottom panel: zoomed on the 8-tone region with each nominal frequency labeled via a
leader line, positioned to avoid overlapping at this spacing.


In [ ]:
def plot_multitone_full_fft(tag, step=0):
    meta, I_arr, Q_arr = load_multitone_run(tag)
    n_tones = meta["n_tones"]
    fs = meta["decimated_fs_mhz"]
    colors = plt.cm.viridis(np.linspace(0, 1, n_tones))

    I_concat = I_arr[step].ravel()
    n = len(I_concat)
    hann = np.hanning(n)
    spec = np.abs(np.fft.rfft((I_concat - I_concat.mean()) * hann))
    freqs = np.fft.rfftfreq(n, d=1 / fs)

    fig, axes = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={"height_ratios": [1, 1.3]})

    axes[0].plot(freqs, spec, lw=1.1, color="steelblue")
    axes[0].set_xlim(0, fs / 2)
    axes[0].set_xlabel("freq [MHz]")
    axes[0].set_ylabel("|FFT|")
    axes[0].set_title(f"Stage D ({tag}): full FFT of one interleaved 8-tone step "
                       f"(step={step}), full Nyquist range")
    axes[0].axvspan(100, 119, color="orange", alpha=0.15)

    ax = axes[1]
    ax.plot(freqs, spec, lw=1.5, color="steelblue", zorder=1)
    ymax = spec.max()
    label_xs = np.linspace(100, 119, n_tones)
    for k in range(n_tones):
        f0 = meta["tone_starts_mhz"][k]
        lx = label_xs[k]
        ax.axvline(f0, color=colors[k], ls="--", lw=1, zorder=2)
        ax.annotate(f"tone{k}: {f0:.0f}MHz", xy=(f0, ymax * 1.0), xytext=(lx, ymax * (1.15 + 0.16 * (k % 2))),
                    ha="center", fontsize=9, color=colors[k], fontweight="bold",
                    arrowprops=dict(arrowstyle="-", color=colors[k], lw=0.8))
    ax.set_xlim(98, 121)
    ax.set_ylim(0, ymax * 1.5)
    ax.set_xlabel("freq [MHz]")
    ax.set_ylabel("|FFT|")
    ax.set_title("zoomed on the 8-tone region")
    fig.tight_layout()
    return meta


_ = plot_multitone_full_fft("final")


## Predicted spectrum for a genuine continuous multi-tone capture (simulated, not measured)

Every plot above uses Stage D's actual per-tone captures, each shielded by `trig_time` to
land in the clean region -- so none of them show what a truly *continuous* capture spanning
tone boundaries would look like. Stage E already answered this for 2 tones: switching
periodically produces sidebands spaced at the hop rate around each carrier (textbook TDM, not
a novel effect). This extends that finding to the full 8-tone case **by simulation** (no
hardware involved) -- construct the idealized waveform directly from Stage D's actual
parameters (`SEG_LEN_US=0.85us` dwell, dummypulse flush gap, `RO_CH=10`'s 307.2MHz decimated
rate), assuming a phase-continuous DDS across hops (matching the empirical finding that
same-frequency boundary transitions are seamless), then FFT it.

**Prediction**: the 8 real carriers remain visible as local peaks at their exact nominal
frequencies, but `f_hop = 1/(8*0.85us) ≈ 0.146MHz` is far finer than the 1MHz tone spacing --
so instead of a few isolated Stage-E-style sideband pairs, the gaps between tones fill in with
a dense forest of closely-packed sidebands, more like a noisy plateau with 8 slightly-taller
peaks poking through than 8 clean isolated lines. This is the same TDM-sideband mechanism as
Stage E, just denser (8 slots/cycle instead of 2).


In [ ]:
def simulate_predicted_multitone_fft(tone_starts_mhz, seg_len_us, dummy_len_us,
                                      decimated_fs_mhz, n_cycles=12):
    """Pure simulation (no hardware) -- idealized continuous capture spanning
    n_cycles full round-robins of tone_starts_mhz, each held for seg_len_us
    then a dummy_len_us zero-gain gap, sampled at decimated_fs_mhz. Phase
    accumulates continuously across hops (no reset) -- matches the empirical
    finding that same-frequency boundary transitions are seamless."""
    n_tones = len(tone_starts_mhz)
    cycle_us = n_tones * (seg_len_us + dummy_len_us)
    f_hop = 1 / cycle_us
    dt = 1 / decimated_fs_mhz

    seq = []
    t_cursor = 0.0
    for _ in range(n_cycles):
        for f0 in tone_starts_mhz:
            seq.append((t_cursor, t_cursor + seg_len_us, f0))
            t_cursor += seg_len_us
            seq.append((t_cursor, t_cursor + dummy_len_us, None))
            t_cursor += dummy_len_us

    t = np.arange(0, t_cursor, dt)
    I = np.zeros_like(t)
    phase_acc = 0.0
    for (t0, t1, f0) in seq:
        idx0, idx1 = int(t0 / dt), int(t1 / dt)
        if idx1 <= idx0:
            continue
        seg_t = t[idx0:idx1] - t[idx0]
        if f0 is None:
            I[idx0:idx1] = 0.0
        else:
            I[idx0:idx1] = np.cos(2 * np.pi * f0 * seg_t + phase_acc)
            phase_acc = (phase_acc + 2 * np.pi * f0 * (seg_t[-1] + dt)) % (2 * np.pi)

    n = len(I)
    hann = np.hanning(n)
    spec = np.abs(np.fft.rfft((I - I.mean()) * hann))
    freqs = np.fft.rfftfreq(n, d=dt)
    return t, I, freqs, spec, f_hop, cycle_us


tone_starts_mhz = [106.0, 107.0, 108.0, 109.0, 110.0, 111.0, 112.0, 113.0]
n_tones = len(tone_starts_mhz)
t_sim, I_sim, freqs_sim, spec_sim, f_hop, cycle_us = simulate_predicted_multitone_fft(
    tone_starts_mhz, seg_len_us=0.85, dummy_len_us=3 / 599.04, decimated_fs_mhz=307.2, n_cycles=12)
print(f"cycle_us={cycle_us:.4f}, f_hop={f_hop:.4f}MHz (predicted sideband spacing)")

colors = plt.cm.viridis(np.linspace(0, 1, n_tones))

fig, axes = plt.subplots(3, 1, figsize=(13, 11))
n_show = int(2 * cycle_us * 307.2)
axes[0].plot(t_sim[:n_show], I_sim[:n_show], lw=0.7)
axes[0].set_xlabel("time [us]")
axes[0].set_ylabel("I (simulated)")
axes[0].set_title(f"Simulated waveform, first 2 cycles ({n_tones} tones/cycle, 0.85us dwell each)")

axes[1].plot(freqs_sim, spec_sim, lw=0.8, color="steelblue")
axes[1].set_xlim(0, 307.2 / 2)
axes[1].set_xlabel("freq [MHz]")
axes[1].set_ylabel("|FFT| (predicted)")
axes[1].set_title("Predicted full-Nyquist spectrum")
for f0 in tone_starts_mhz:
    axes[1].axvline(f0, color="red", ls=":", lw=0.6, alpha=0.6)

axes[2].plot(freqs_sim, spec_sim, lw=1.2, color="steelblue")
axes[2].set_xlim(100, 120)
for k, f0 in enumerate(tone_starts_mhz):
    axes[2].axvline(f0, color=colors[k], ls="--", lw=1.3)
    axes[2].annotate(f"tone{k}\n{f0:.0f}", xy=(f0, spec_sim.max() * 0.95), ha="center",
                     fontsize=8, color=colors[k], fontweight="bold")
axes[2].set_xlabel("freq [MHz]")
axes[2].set_ylabel("|FFT| (predicted)")
axes[2].set_title(f"Zoomed: expect the 8 carriers PLUS sidebands spaced by f_hop={f_hop:.3f}MHz around each")
fig.tight_layout()


# Stage G design -- engineered AM-sideband comb (math validated, hardware UNTESTED)

Every multitone approach so far (Stage D's time-multiplexing, Stage E's TDM-comb hypothesis)
produces the 8 tones *sequentially*, with switching artifacts as an unavoidable side effect.
This is a different idea: instead of hopping the carrier frequency, keep the DDS carrier
**fixed** at the center (109.5MHz) and multiply it by a custom **gain envelope** built from
exactly 4 cosines at 0.5, 1.5, 2.5, 3.5MHz. Real amplitude modulation of a real carrier
produces both upper *and* lower sidebands for each modulation component, so
`109.5 ± {0.5,1.5,2.5,3.5} = 106,107,108,109,110,111,112,113` — all 8 target tones, genuinely
**simultaneous** (one instant of the waveform contains all 8), by direct Fourier construction
rather than as switching debris. Sweeping the whole comb is then just sweeping the *carrier*
register by 1MHz with the exact same cascade mechanism as Stage C — the envelope (which
encodes the fixed relative spacing) never changes.

The cell below is a pure-math check (no hardware) of whether this actually produces a clean
comb before committing to writing any QICK code. It does: **~78dB down** everywhere except
the 8 target frequencies (peak amplitude ~29458 at every target, ~1500x more than that
already, and versus ~3.6 max elsewhere in the whole 100-120MHz band) — dramatically cleaner
than anything switching-based produced this session.

**The hardware implementation is untested** — `python/stageG_am_comb_sweep.py` and the
matching section in `FreqSweep.ipynb` implement this (custom `add_envelope`, `style="arb"`,
same `read_wmem`/`inc_reg`/`write_wmem` cascade as Stage C applied to the carrier register)
but the board was disconnected before any of it could be run. See that script's docstring for
the specific API points that need verification first (exact `add_envelope` signature, whether
the register cascade behaves identically for `arb`- vs `const`-style pulses, `trig_time` for
this pulse type, and the I/Q convention for a real-valued envelope).


In [ ]:
def check_am_comb_math(f_center=109.5, mod_offsets=(0.5, 1.5, 2.5, 3.5),
                        dac_fs_mhz=9584.64, n_periods=50):
    """Pure-math validation (no hardware) that AM-modulating a fixed carrier
    with exactly these 4 cosines produces exactly the 8 target sidebands and
    nothing else of significance."""
    t_mod_us = 1.0 / mod_offsets[0]
    n_env = int(round(t_mod_us * dac_fs_mhz))
    t_env = np.arange(n_env) / dac_fs_mhz
    m = sum(np.cos(2 * np.pi * f * t_env) for f in mod_offsets)
    m /= np.max(np.abs(m))

    carrier_t = np.arange(n_periods * n_env) / dac_fs_mhz
    env_repeated = np.tile(m, n_periods)
    x = env_repeated * np.cos(2 * np.pi * f_center * carrier_t)

    n = len(x)
    hann = np.hanning(n)
    spec = np.abs(np.fft.rfft((x - x.mean()) * hann))
    freqs = np.fft.rfftfreq(n, d=1 / dac_fs_mhz)

    targets = sorted([f_center + o for o in mod_offsets] + [f_center - o for o in mod_offsets])
    fig, axes = plt.subplots(2, 1, figsize=(12, 7))
    axes[0].plot(carrier_t[:3 * n_env], x[:3 * n_env], lw=0.6)
    axes[0].plot(carrier_t[:3 * n_env], env_repeated[:3 * n_env], color="red", lw=1.2, label="gain envelope m(t)")
    axes[0].plot(carrier_t[:3 * n_env], -env_repeated[:3 * n_env], color="red", lw=1.2)
    axes[0].legend(fontsize=8)
    axes[0].set_xlabel("time [us]")
    axes[0].set_title("carrier x engineered gain envelope, first 3 modulation periods")

    axes[1].plot(freqs, spec, lw=1.0, color="steelblue")
    axes[1].set_xlim(f_center - 10, f_center + 10)
    colors = plt.cm.viridis(np.linspace(0, 1, len(targets)))
    for k, f0 in enumerate(targets):
        axes[1].axvline(f0, color=colors[k], ls="--", lw=1)
        axes[1].annotate(f"{f0:.0f}", xy=(f0, spec.max() * 0.9), ha="center",
                         fontsize=8, color=colors[k], fontweight="bold")
    axes[1].set_xlabel("freq [MHz]")
    axes[1].set_ylabel("|FFT|")
    axes[1].set_title("Predicted spectrum: engineered AM-sideband comb (not switching-based)")
    fig.tight_layout()

    peak_vals = {f0: spec[np.argmin(np.abs(freqs - f0))] for f0 in targets}
    mask = np.zeros_like(freqs, dtype=bool)
    for f0 in targets:
        mask |= np.abs(freqs - f0) < 0.2
    band = (freqs > f_center - 10) & (freqs < f_center + 10) & ~mask
    print("peak amplitudes at targets:", {f0: round(v, 1) for f0, v in peak_vals.items()})
    print(f"max elsewhere in {f_center-10:.0f}-{f_center+10:.0f}MHz (excluding targets +/-0.2MHz): "
          f"{spec[band].max():.2f}")
    print(f"suppression ratio: {min(peak_vals.values()) / spec[band].max():.0f}x "
          f"({20*np.log10(min(peak_vals.values()) / spec[band].max()):.1f}dB)")
    return targets, peak_vals


_ = check_am_comb_math()


## Findings -- Stage G: math validated, hardware NOT YET ATTEMPTED

The engineered AM-sideband comb produces exactly the 8 target frequencies at **~78dB** above
everything else in the band (peak amplitude ~29458 at every target vs. max 3.6 elsewhere) --
by a wide margin the cleanest multitone spectrum of any approach considered this session
(Stage D's real switching comb and Stage E's TDM sidebands were both busy/dense by
comparison, not because they were done wrong, but because switching-based generation
inherently trades off envelope width against comb density — see the "envelope vs. comb"
explainer and the N-periods-per-tone scaling analysis above). This approach sidesteps that
tradeoff entirely by constructing the sidebands directly via Fourier synthesis of the gain
envelope, rather than as a byproduct of frequency-hopping.

**Not yet real**: this is a validated *idea*, not a validated *implementation*. The hardware
attempt (`python/stageG_am_comb_sweep.py`, `FreqSweep.ipynb`) is written but explicitly marked
untested — the board was disconnected before any of it could be run. Specific open questions
before trusting it: exact `add_envelope()` signature, whether the same `read_wmem`/
`inc_reg`/`write_wmem` register-cascade sweep (proven for `const`-style pulses in Stage C)
behaves identically for `arb`-style pulses, and the correct `trig_time`/pipeline-transient
behavior for this pulse type (never characterized -- `boundary_capture.py`'s measurement was
`const`-style only).
